# 1. Recover the digit distribution

How many images of each digit are held across the federation?
Write the local and global counting functions, check them on a tiny example,
then run your code on the three deployed SuperNodes.

Complete [setup](../docs/setup.md) and download the MNIST app into `work/` first.
The architecture and first training run are guided demonstrations; this
notebook starts with the coding exercise. You can work alone or in a small group.


In [ ]:
from pathlib import Path
import sys

ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "notebooks/workshop.py").is_file() and (path / "configs/train.toml").is_file()
)
sys.path.insert(0, str(ROOT / "notebooks"))
from workshop import analytics_student, run_analytics

RUN_ON_FEDERATION = False  # Set True after logging in and accepting the invitation.
FEDERATION = "@charles/flta"


## Write two functions

Open `work/paris-mnist-fl/mnist_fl/analytics_student.py` in your editor:

- `local_histogram(labels)` returns ten Python integer counts, indexed by digit
- `merge_histograms(reports)` combines the distinct reports' `counts` lists

Include a zero for any digit absent from a node. A reply needs ten counts,
without labels, image identifiers or images. See the [function contracts](../docs/analytics.md#1-how-many-of-each-digit).

Save your changes, then run the check below. It loads your edited source each
time. A `NotImplementedError` means a required student function is unfinished.


In [ ]:
student = analytics_student(ROOT)
a = student.local_histogram([0, 0, 3, 9])
b = student.local_histogram([3, 3, 8])
assert a == [2, 0, 0, 1, 0, 0, 0, 0, 0, 1]
assert student.local_histogram([]) == [0] * 10
assert all(type(value) is int for value in a + b)
pooled = student.merge_histograms([{"counts": a}, {"counts": b}])
assert pooled == [2, 0, 0, 3, 0, 0, 0, 0, 1, 1]
assert all(type(value) is int for value in pooled)
print("Your histogram functions passed the tiny checks")


## Run on the deployed federation

Predict the most frequent digit before you run. The app uses a fixed subset
of 6,000 MNIST **training** images, assigned to nodes through Flower Datasets.
Its coordinator combines summaries without loading MNIST.

The next cell previews the command until you set `RUN_ON_FEDERATION = True`
in the setup cell and rerun it. After membership is ready, enable it and run
the submission cell. Your saved app source is packaged and submitted.

Both runs select `analytics-implementation = "student"`. They keep
`save-output = false` and `publish-release = false`; a copy of each run's
config and terminal stream is saved locally under `outputs/notebook-runs/`.


In [ ]:
iid_run = run_analytics(
    ROOT, "histogram", "iid", execute=RUN_ON_FEDERATION, federation=FEDERATION
)


## Change the data split, keep the population

Repeat with label-skewed partitions, which give nodes different digit
distributions. Before running, decide which parts of the result should change
and which should stay the same. This changes data
assignment in the app; the three deployed nodes stay connected.


In [ ]:
skew_run = run_analytics(
    ROOT, "histogram", "label-skew", execute=RUN_ON_FEDERATION, federation=FEDERATION
)


In [ ]:
if iid_run is None or skew_run is None:
    print("Comparison pending: complete both deployed runs first")
else:
    iid, skew = iid_run["summary"], skew_run["summary"]
    assert len(iid["counts"]) == len(skew["counts"]) == 10
    assert sum(iid["counts"]) == sum(skew["counts"]) == 6000
    assert iid["counts"] == skew["counts"]
    print("digit | global count")
    for digit, count in enumerate(iid["counts"]):
        print(f"{digit:>5} | {count}")
    print("Complete deployed histograms agree")


## Keep your observations

Edit this cell with your results:

| Observation | Your result |
| --- | --- |
| Prediction | |
| IID run ID | |
| Label-skew run ID | |
| Most frequent digit and its count | |
| Something you checked or fixed | |

Which function executed on each SuperNode? Which executed at the coordinator?
Why can the local distributions change while the global histogram stays fixed?

**Checkpoint:** your two functions pass local checks and produce matching
complete histograms on the deployed federation. The coordinator sees local
count summaries; this exercise does not add secure aggregation or differential privacy.

Next: [ink statistics](02_ink_statistics.ipynb), or choose a
[histogram extension](../docs/extensions.md#after-the-histogram).
